# DP2 — DDF consolidated visit table via Butler

**Author:** dagoret
**Creation Date:** 2026-07-24
**Last revision:** 2026-07-24 (rewritten)
**version:** v1

## Purpose

Query the **Butler Gen3** repository to find and extract the **final consolidated
per-visit table** for DP2 (the one carrying visit-level metadata such as sky
background and seeing), restrict it to the **Deep Drilling Fields (DDF)**, and
save it in a format directly usable by the downstream `skysurvey` SN simulation
notebook (`01_simSNIaDDF_frSimpleDP2visits.ipynb`), which expects (among others)
`ra`, `dec`, `mjd`, `band`, `exptime`, `airmass`, `seeing`, `skybg`/`maglim` columns.

**Why this notebook is structured this way.** The exact dataset-type name that
carries the consolidated visit table can differ between DRP runs/versions
(`visitTable`, `ccdVisitTable`, `visitSummary`, ...), and the ones with the
richest per-visit content (seeing, sky background) are often **per-detector**
tables that need aggregating to one row per visit. Rather than hardcoding a
guess, **Section 3 discovers the available dataset types**, **Section 4 probes
the most likely candidates and prints their columns**, and only then does
**Section 5** load the one you pick. No TAP service is used (not yet available
for DP2 at USDF).

---
## 0. Imports

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import os
import re
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u
from astropy.time import Time
from astropy.io import ascii as astropy_ascii

import lsst
from lsst.daf.butler import Butler
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

print(f"Matplotlib : {matplotlib.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Imports OK")

In [ ]:
mpl.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "font.size": 14,
        "axes.titlesize": 18,
        "axes.labelsize": 16,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 14,
        "legend.title_fontsize": 15,
        "figure.titlesize": 20,
    }
)

---
## 1. User Configuration

In [ ]:
# --- output directories ---------------------------------------------------
NB_TAG = "DP2_DDF_VISITTABLE_BUTLER_01"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

# Where the final visit table (all DDF, all bands) is written.
# Both a raw CSV (all discovered columns) and an ECSV formatted like
# ``dp2_visits_table_with_iq.ecsv`` (the file the skysurvey notebook reads)
# are produced in Section 9.
OUTPUT_CSV = os.path.join(DIR_DATA, "dp2_ddf_consolidated_visits.csv")
OUTPUT_ECSV = os.path.join(DIR_DATA, "dp2_visits_table_with_iq.ecsv")

In [ ]:
# --- Butler repo / collections ---------------------------------------------
REPO = "dp2_prep"

COLLECTIONS = [
    "LSSTCam/defaults",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

SKYMAP_NAME = "lsst_cells_v2"
INSTRUMENT = "LSSTCam"
WHERE_CLAUSE_INSTRUMENT = f"instrument = '{INSTRUMENT}'"
DATE_START = 20250415
WHERE_CLAUSE_DATE = WHERE_CLAUSE_INSTRUMENT + f" and day_obs >= {DATE_START}"

# --- DDF field centers (deg) -------------------------------------------------
DDF_COORDS = {
    "COSMOS": (150.119, +2.206),
    "ECDFS": (53.125, -28.100),
    "ELAIS-S1": (9.450, -44.000),
    "XMM-LSS": (35.708, -4.750),
    "EDFS-a": (58.900, -49.315),
    "EDFS-b": (63.600, -47.600),
    "EDFS": (61.240, -48.423),
    "M49": (187.400, +8.000),
}

# Search radius used both for tract discovery and for the position-based
# DDF membership fallback (Section 6). ~2.0-2.1 deg comfortably covers the
# Rubin FoV (3.5 deg diagonal / 2 => ~1.75 deg half-diagonal) plus margin.
TRACT_SEARCH_RADIUS_DEG = 1.8
DDF_MATCH_RADIUS_DEG = 2.1

print(f"Repo        : {REPO}")
print(f"Collections : {COLLECTIONS}")
print(f"Where clause (instrument) : {WHERE_CLAUSE_INSTRUMENT}")
print(f"Where clause (date)       : {WHERE_CLAUSE_DATE}")

---
## 2. Butler and SkyMap

In [ ]:
butler = Butler(REPO, collections=COLLECTIONS)
registry = butler.registry
print("Butler OK")

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME)
except Exception as e:
    print(f"Could not fetch skyMap '{SKYMAP_NAME}': {e}")
    raise
print(f"SkyMap '{SKYMAP_NAME}' : {len(skymap)} tracts")

---
## 3. Dataset Type Discovery

List every dataset type whose name contains `visit`, together with its
**dimensions** and **storage class**. This tells us:
- which ones are single consolidated tables (no `visit` dimension — one
  dataset covers the whole collection, e.g. typically `visitTable` /
  `ccdVisitTable`);
- which ones are per-visit products you'd need to loop over
  (dimension includes `visit`, e.g. typically `visitSummary`, an
  `ExposureCatalog` with one row per detector).

Run this once per repo/collection set — the result tells you which name to
use in Section 4/5.

In [ ]:
all_datasets = list(registry.queryDatasetTypes())
visit_datasets = sorted(
    [dt for dt in all_datasets if "visit" in dt.name.lower()], key=lambda x: x.name
)

print(f"{len(visit_datasets)} dataset types with 'visit' in the name:\n")
for dt in visit_datasets:
    print(f"  {dt.name:35s} dims={sorted(dt.dimensions.names):40} storageClass={dt.storageClass.name}")

---
## 4. Probe Candidate Consolidated Visit Tables

We try a short list of the names most commonly used in Rubin DRP runs for
the **consolidated** (survey-wide) visit table, in priority order:

1. `visitTable` — per-visit, one row per visit (columns are usually already
   averaged over detectors where relevant).
2. `ccdVisitTable` — per-detector-visit, one row per (visit, detector); has
   the richest set of per-detector conditions (often includes `seeing`,
   `skyBg`, `skyNoise`, `zeroPoint`, `psfSigma`) and needs aggregating to
   visit level (done in Section 5).
3. `visitSummary` — an `ExposureCatalog`, one dataset **per visit**
   (dimensions include `visit`), one row per detector inside. Richest but
   requires iterating over all visit dataIds — slow for the full survey, so
   we only probe a single visit here to inspect the schema.

For each, we fetch a small sample and print the columns, flagging any that
look like seeing / sky-background / zeropoint / limiting-magnitude fields so
you can immediately tell which dataset type has what you need.

In [ ]:
KEYWORDS = {
    "seeing/psf": ["seeing", "fwhm", "psfsigma", "psf_fwhm"],
    "sky background": ["skybg", "sky_bg", "skybackground", "sky_brightness", "skymag"],
    "sky noise": ["skynoise", "sky_noise"],
    "zeropoint": ["zeropoint", "zero_point", "magzero", "^zp$"],
    "limiting mag": ["maglim", "fivesigmadepth", "^m5$", "depth"],
    "airmass": ["airmass"],
    "mjd": ["mjd"],
}


def flag_columns(columns):
    """Return {quantity: [matching column names]} for the KEYWORDS dict above."""
    hits = {}
    for label, patterns in KEYWORDS.items():
        matches = [
            c for c in columns if any(re.search(p, c, flags=re.IGNORECASE) for p in patterns)
        ]
        if matches:
            hits[label] = matches
    return hits


def to_dataframe(obj):
    """Convert a Butler-returned catalog (DataFrame / astropy Table / afw catalog) to pandas."""
    if isinstance(obj, pd.DataFrame):
        return obj
    if hasattr(obj, "to_pandas"):
        return obj.to_pandas()
    if hasattr(obj, "asAstropy"):
        return obj.asAstropy().to_pandas()
    if isinstance(obj, Table):
        return obj.to_pandas()
    raise TypeError(f"Cannot convert {type(obj)} to a DataFrame.")


print("Helpers defined: flag_columns(), to_dataframe()")

In [ ]:
CANDIDATE_VISIT_TABLE_TYPES = ["visitTable", "ccdVisitTable"]

probe_results = {}

for name in CANDIDATE_VISIT_TABLE_TYPES:
    print(f"\n=== Probing '{name}' ===")
    try:
        raw = butler.get(name, instrument=INSTRUMENT)
        df_probe = to_dataframe(raw)
        probe_results[name] = df_probe
        print(f"OK -- shape={df_probe.shape}")
        print(f"columns: {list(df_probe.columns)}")
        hits = flag_columns(df_probe.columns)
        print("Keyword matches:")
        for label, cols in hits.items():
            print(f"    {label:16s} -> {cols}")
        if not hits:
            print("    (no seeing/sky/zeropoint-like column found by keyword match)")
    except Exception as e:
        print(f"FAILED: {e}")

In [ ]:
# visitSummary is dimensioned by (instrument, visit): probe a single visit
# to inspect its schema without pulling the whole survey.
print("=== Probing 'visitSummary' (single visit) ===")
try:
    refs_vs = list(
        registry.queryDatasets("visitSummary", where=WHERE_CLAUSE_DATE, limit=1)
    )
    if refs_vs:
        vs_raw = butler.get(refs_vs[0])
        df_vs_probe = to_dataframe(vs_raw)
        print(f"OK -- 1 visit, {df_vs_probe.shape[0]} detector rows, {df_vs_probe.shape[1]} columns")
        print(f"columns: {list(df_vs_probe.columns)}")
        hits = flag_columns(df_vs_probe.columns)
        print("Keyword matches:")
        for label, cols in hits.items():
            print(f"    {label:16s} -> {cols}")
    else:
        print("No visitSummary datasets found for this where-clause.")
except Exception as e:
    print(f"FAILED: {e}")

---
## 5. Load the Chosen Consolidated Visit Table

**Action needed:** look at the Section 4 output, decide which dataset type
actually carries `seeing` + sky-background-like columns, and set
`VISIT_DATASET_TYPE` below accordingly. `ccdVisitTable` is the most likely
candidate to carry both directly (per-detector); if you pick a per-detector
table, `load_consolidated_visits()` aggregates it to one row per visit
(mean over detectors for numeric conditions, first value for
ra/dec/band/mjd/exptime).

In [ ]:
# TODO: confirm/edit after reading the Section 4 printout.
VISIT_DATASET_TYPE = "ccdVisitTable"  # e.g. "visitTable" or "ccdVisitTable"

# Columns to average across detectors when aggregating a per-detector table
# to one row per visit. Extend this list based on what Section 4 found.
AGG_MEAN_PATTERNS = ["seeing", "fwhm", "psfsigma", "skybg", "sky_bg", "skynoise",
                     "zeropoint", "zero_point", "airmass", "zenithdistance"]
# Columns that are constant across detectors within a visit: just take the first.
AGG_FIRST_PATTERNS = ["ra", "dec", "band", "filter", "mjd", "exptime", "target",
                      "day_obs", "expmidpt", "obsstart", "airmass", "azimuth", "altitude"]


def load_consolidated_visits(butler, dataset_type_name, instrument=INSTRUMENT):
    """Fetch a consolidated visit(-like) table and return a per-visit DataFrame.

    If the table already has one row per visit, it is returned as-is
    (after locating the visit-id column). If it has one row per
    (visit, detector), it is aggregated to one row per visit.
    """
    raw = butler.get(dataset_type_name, instrument=instrument)
    df = to_dataframe(raw)

    visit_col = next(
        (c for c in df.columns if c.lower() in ("visitid", "visit_id", "visit")), None
    )
    if visit_col is None and df.index.name and "visit" in df.index.name.lower():
        df = df.reset_index()
        visit_col = df.columns[0]
    if visit_col is None:
        raise ValueError(f"Could not find a visit-id column in '{dataset_type_name}'.")

    detector_col = next((c for c in df.columns if c.lower() == "detector"), None)
    if detector_col is None:
        print(f"'{dataset_type_name}' already looks visit-level (no detector column).")
        return df, visit_col

    print(f"'{dataset_type_name}' is per-detector -- aggregating to one row per visit.")
    mean_cols = [c for c in df.columns
                 if any(re.search(p, c, re.IGNORECASE) for p in AGG_MEAN_PATTERNS)]
    first_cols = [c for c in df.columns
                  if c not in mean_cols and any(re.search(p, c, re.IGNORECASE) for p in AGG_FIRST_PATTERNS)]
    agg = {c: "mean" for c in mean_cols}
    agg.update({c: "first" for c in first_cols})
    df_visit = df.groupby(visit_col, as_index=False).agg(agg)
    print(f"Aggregated: {df.shape[0]} detector-visit rows -> {df_visit.shape[0]} visits")
    print(f"  averaged (mean over detectors): {mean_cols}")
    print(f"  kept as-is (first detector)    : {first_cols}")
    return df_visit, visit_col


df_visits_raw, VISIT_ID_COL = load_consolidated_visits(butler, VISIT_DATASET_TYPE)
print(f"\nFinal consolidated visit table: {df_visits_raw.shape[0]} visits, "
      f"{df_visits_raw.shape[1]} columns.")
df_visits_raw.head()

In [ ]:
df_visits_raw.to_csv(OUTPUT_CSV, index=False)
print(f"Saved unfiltered consolidated visit table -> {OUTPUT_CSV}")

---
## 6. Restrict to the Deep Drilling Fields

Two complementary strategies, applied in order:

1. **By target/field name**, if the table carries a `target` /
   `science_program` / `observation_reason`-like string column: match
   against the DDF names.
2. **By sky position** (always computed, used as the primary method when no
   usable name column exists, and as a cross-check otherwise): angular
   separation to each DDF center, vectorized with `astropy.coordinates`
   (no python-level loop over visits).

In [ ]:
ra_col = next(c for c in df_visits_raw.columns if c.lower() in ("ra", "fieldra", "boresightra"))
dec_col = next(c for c in df_visits_raw.columns if c.lower() in ("dec", "decl", "fielddec", "boresightdec"))
band_col = next(c for c in df_visits_raw.columns if c.lower() in ("band", "physical_filter", "filter"))
name_col = next(
    (c for c in df_visits_raw.columns if c.lower() in ("target", "science_program", "observation_reason")),
    None,
)
print(f"ra_col={ra_col!r}  dec_col={dec_col!r}  band_col={band_col!r}  name_col={name_col!r}")

In [ ]:
ddf_names = list(DDF_COORDS.keys())
ddf_centers = SkyCoord(
    ra=[c[0] for c in DDF_COORDS.values()] * u.deg,
    dec=[c[1] for c in DDF_COORDS.values()] * u.deg,
)
visit_coords = SkyCoord(ra=df_visits_raw[ra_col].values * u.deg, dec=df_visits_raw[dec_col].values * u.deg)

# separation matrix: (n_visits, n_ddf), vectorized (no per-row python loop)
sep_matrix = np.array([visit_coords.separation(c).deg for c in ddf_centers]).T  # (n_visits, n_ddf)
nearest_idx = sep_matrix.argmin(axis=1)
nearest_sep = sep_matrix.min(axis=1)

df_visits_raw = df_visits_raw.copy()
df_visits_raw["ddf_name_by_position"] = np.array(ddf_names)[nearest_idx]
df_visits_raw["ddf_separation_deg"] = nearest_sep
is_ddf_by_position = nearest_sep <= DDF_MATCH_RADIUS_DEG

if name_col is not None:
    is_ddf_by_name = df_visits_raw[name_col].astype(str).str.contains(
        "DDF|COSMOS|ELAIS|XMM|ECDFS|EDFS|M49", case=False, na=False
    )
    is_ddf = is_ddf_by_name | is_ddf_by_position
    print(f"DDF by name     : {int(is_ddf_by_name.sum())} visits")
    print(f"DDF by position : {int(is_ddf_by_position.sum())} visits")
else:
    is_ddf = is_ddf_by_position
    print(f"No target-name column found -- using position match only: "
          f"{int(is_ddf_by_position.sum())} visits")

df_ddf_visits = df_visits_raw[is_ddf].copy()
df_ddf_visits["ddf_name"] = df_ddf_visits["ddf_name_by_position"]
print(f"\nTotal DDF visits kept: {len(df_ddf_visits)} / {len(df_visits_raw)}")
print(df_ddf_visits["ddf_name"].value_counts())

---
## 7. Tract / Patch Geometrical Mapping for the DDF

In [ ]:
def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=1.8):
    cos_dec = max(np.cos(np.deg2rad(dec_deg)), 0.01)
    step = 0.35
    found_ids = set()
    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s = ra_deg + dra / cos_dec
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = SpherePoint(ra_s * degrees, dec_s * degrees)
            try:
                found_ids.add(skymap.findTract(sp).tract_id)
            except Exception:
                pass
    return sorted(found_ids)


ddf_geometry_mapping = []
for name, (ra, dec) in DDF_COORDS.items():
    center = geom.SpherePoint(ra * geom.degrees, dec * geom.degrees)
    tract_ids = find_tracts_for_coord(skymap, ra, dec, radius_deg=TRACT_SEARCH_RADIUS_DEG)

    r_ang = TRACT_SEARCH_RADIUS_DEG * geom.degrees
    coord_list = [
        center,
        center.offset(r_ang, 0 * geom.degrees),
        center.offset(-r_ang, 0 * geom.degrees),
        center.offset(0 * geom.degrees, r_ang),
        center.offset(0 * geom.degrees, -r_ang),
    ]

    for t_id in tract_ids:
        tract_info = skymap.generateTract(t_id)
        try:
            patch_info_list = tract_info.findPatchList(coord_list)
        except Exception as e:
            print(f"Skipping tract {t_id} for {name}: {str(e)[:60]}")
            continue
        for patch_info in patch_info_list:
            ix, iy = patch_info.getIndex()
            n_patches_x = tract_info.getNumPatches()[0]
            ddf_geometry_mapping.append(
                {
                    "ddf_name": name,
                    "tract": t_id,
                    "patch": f"{ix},{iy}",
                    "patch_id": iy * n_patches_x + ix,
                }
            )

df_ddf_map = pd.DataFrame(ddf_geometry_mapping).drop_duplicates()
print(f"Mapping done: {len(df_ddf_map)} tract/patch combinations found across {len(DDF_COORDS)} DDFs.")
for name in DDF_COORDS:
    tracts = sorted(df_ddf_map.loc[df_ddf_map.ddf_name == name, "tract"].unique())
    print(f"  {name:10s} tracts: {tracts}")

---
## 8. Diagnostics: Visits per Night, per Band, per DDF

In [ ]:
band_order = ["u", "g", "r", "i", "z", "y"]
band_colors = {
    "u": "#8000ff", "g": "#228b22", "r": "#ff0000",
    "i": "#ff8c00", "z": "#708090", "y": "#000000",
}

mjd_col = next((c for c in df_ddf_visits.columns if "mjd" in c.lower()), None)
print(f"Using MJD column: {mjd_col!r}")

df_plot_all = df_ddf_visits.copy()
df_plot_all["band_clean"] = df_plot_all[band_col].astype(str).str[0]
df_plot_all["night"] = np.floor(df_plot_all[mjd_col]).astype(int)

tmin = np.floor(df_plot_all[mjd_col].min()) - 5
tmax = np.ceil(df_plot_all[mjd_col].max()) + 5

fig, axes = plt.subplots(
    len(DDF_COORDS), 1, figsize=(15, 3.5 * len(DDF_COORDS)),
    sharex=False, gridspec_kw={"hspace": 0.05}, constrained_layout=True,
)
if len(DDF_COORDS) == 1:
    axes = [axes]

for ax, ddf_name in zip(axes, DDF_COORDS.keys()):
    df_plot = df_plot_all[df_plot_all["ddf_name"] == ddf_name]
    ax.set_xlim(tmin, tmax)
    if df_plot.empty:
        ax.set_title(f"DDF: {ddf_name} | Total: 0 visits", fontweight="bold")
        continue

    total_visits = len(df_plot)
    occ_map = df_plot.groupby(["night", "band_clean"]).size().unstack(fill_value=0)
    for b in band_order:
        if b not in occ_map.columns:
            occ_map[b] = 0
    occ_map = occ_map[band_order]

    bottom = np.zeros(len(occ_map))
    for band in band_order:
        n_band = occ_map[band].sum()
        ax.bar(occ_map.index, occ_map[band], bottom=bottom,
               label=f"band {band} (n={n_band})", color=band_colors[band], width=0.8)
        bottom += occ_map[band].values

    ax.set_ylabel("Visits / Night", fontsize=11)
    ax.set_title(f"DDF: {ddf_name} | Total Visits: {total_visits}", fontsize=14,
                 fontweight="bold", pad=35)
    ax.grid(linestyle="--", alpha=0.5)

    ax_top = ax.secondary_xaxis("top", functions=(lambda x: x, lambda x: x))
    tick_locations = np.linspace(tmin, tmax, 10)
    ax_top.set_xticks(tick_locations)
    ax_top.set_xticklabels(
        [Time(t, format="mjd").to_datetime().strftime("%Y-%m-%d") for t in tick_locations],
        rotation=30, ha="left", fontsize=9, color="navy",
    )
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), title=ddf_name, borderaxespad=0)

axes[-1].set_xlabel("MJD (Modified Julian Date)", fontsize=12, fontweight="bold", labelpad=10)
fig.savefig(os.path.join(DIR_FIGS, "ddf_visits_per_night_per_band.pdf"))
fig.savefig(os.path.join(DIR_FIGS, "ddf_visits_per_night_per_band.png"))
plt.show()

---
## 9. Export for `skysurvey`

Save the DDF-restricted consolidated visit table in the ECSV format read by
`01_simSNIaDDF_frSimpleDP2visits.ipynb`. Columns are **renamed to match that
notebook's `CANDIDATES` auto-detection dict directly** (`seeing`, `skyBg`,
in addition to the raw column names), so the skysurvey notebook's Section 2
detects them without manual editing this time.

In [ ]:
RENAME_TO_SKYSURVEY_CANDIDATES = {
    VISIT_ID_COL: "visitId",
    ra_col: "ra",
    dec_col: "dec",
    band_col: "band",
}
# add seeing / sky background aliases if we can identify them by keyword,
# without removing the original column names.
df_export = df_ddf_visits.rename(columns=RENAME_TO_SKYSURVEY_CANDIDATES).copy()

seeing_src = next((c for c in df_export.columns
                    if re.search("seeing|fwhm|psfsigma", c, re.IGNORECASE)), None)
skybg_src = next((c for c in df_export.columns
                   if re.search("skybg|sky_bg|skybackground|sky_brightness", c, re.IGNORECASE)), None)
maglim_src = next((c for c in df_export.columns
                    if re.search("maglim|fivesigmadepth|^m5$|depth", c, re.IGNORECASE)), None)

if seeing_src and seeing_src != "seeing":
    df_export["seeing"] = df_export[seeing_src]
if skybg_src and skybg_src != "skyBg":
    df_export["skyBg"] = df_export[skybg_src]
if maglim_src and maglim_src != "maglim":
    df_export["maglim"] = df_export[maglim_src]

print("seeing source column :", seeing_src)
print("skyBg source column  :", skybg_src)
print("maglim source column :", maglim_src)
if seeing_src is None or skybg_src is None:
    print("\nWARNING: seeing and/or sky-background were not found automatically in "
          f"'{VISIT_DATASET_TYPE}'. Re-check the Section 4 printout and try the other "
          "candidate dataset type (or visitSummary) for VISIT_DATASET_TYPE.")

df_export.to_csv(OUTPUT_CSV.replace(".csv", "_export.csv"), index=False)
Table.from_pandas(df_export).write(OUTPUT_ECSV, format="ascii.ecsv", overwrite=True)
print(f"\nSaved -> {OUTPUT_ECSV}")
print(f"Saved -> {OUTPUT_CSV.replace('.csv', '_export.csv')}")
df_export.head()

---
## 10. Summary / Next Steps

- Section 3–4 tell you which Butler dataset type actually carries the
  consolidated visit information (seeing, sky background) for this DRP run —
  re-run those two sections first if you change `REPO` / `COLLECTIONS`.
- Section 5's `VISIT_DATASET_TYPE` must be set from that discovery; the
  default here (`ccdVisitTable`, aggregated to visit level) is a starting
  guess, not a guarantee for this specific run.
- The DDF-restricted table is in `df_ddf_visits` / `df_export`, and is saved to:
  - `data_DP2_DDF_VISITTABLE_BUTLER_01/dp2_ddf_consolidated_visits.csv` (raw, unfiltered)
  - `data_DP2_DDF_VISITTABLE_BUTLER_01/dp2_ddf_consolidated_visits_export.csv` (DDF-only, renamed)
  - `data_DP2_DDF_VISITTABLE_BUTLER_01/dp2_visits_table_with_iq.ecsv` (skysurvey-ready)
- Copy the `.ecsv` file into
  `SimTtranscientLSSTSurvey/notebooks/01_simulateDP2_SNinDDF/data_in/` and re-run
  `01_simSNIaDDF_frSimpleDP2visits.ipynb` from Section 2 — its column
  auto-detection should now pick up `seeing` and `skyBg` directly.